# 🎓 Student Performance Prediction & Analytics
### **Machine Learning Mini-Project**

**Objective:** Build and evaluate multiple predictive regression models to forecast student final marks based on academic study habits, class attendance, and previous academic foundation.

---
### **Workflow Pipeline:**
1. **Exploratory Data Analysis (EDA)** & Statistical Profiling
2. **Correlation Analysis & Feature Dynamics**
3. **Dataset Splitting & Preprocessing**
4. **Model Training & Comparison** (Linear Regression, Ridge, Random Forest, Gradient Boosting)
5. **Cross-Validation & Error Analysis** ($R^2$, MAE, RMSE)
6. **Residual Diagnostics & Model Validation**
7. **Inference & Scenario Simulation**

In [ ]:
# 1. Import Essential Libraries
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Ensure project root in python path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

# Visual settings
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
print('All libraries loaded successfully!')

## 2. Load and Explore the Dataset

In [ ]:
data_path = PROJECT_ROOT / 'data' / 'student_performance_dataset.csv'
if not data_path.exists():
    data_path = PROJECT_ROOT / 'data' / 'student_performance.csv'

df = pd.read_csv(data_path)
print(f'Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns')
df.head()

In [ ]:
# Statistical summary of the academic parameters
df.describe().round(2)

## 3. Exploratory Data Analysis (EDA) & Visualizations

In [ ]:
# Correlation Heatmap
plt.figure(figsize=(8, 6))
num_df = df.select_dtypes(include=[np.number])
sns.heatmap(num_df.corr(), annot=True, cmap='Blues', fmt='.3f', cbar=True, square=True)
plt.title('Correlation Heatmap with Final Marks', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Distribution of Study Hours, Attendance, and Final Marks
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

sns.histplot(df['Study_Hours'], kde=True, color='#2563eb', ax=axes[0])
axes[0].set_title('Study Hours Distribution')

sns.histplot(df['Attendance'], kde=True, color='#059669', ax=axes[1])
axes[1].set_title('Attendance Distribution (%)')

sns.histplot(df['Final_Marks'], kde=True, color='#7c3aed', ax=axes[2])
axes[2].set_title('Final Marks Distribution')

plt.tight_layout()
plt.show()

In [ ]:
# Scatter plot: Study Hours vs Final Marks colored by Previous Marks
plt.figure(figsize=(10, 6))
scatter = plt.scatter(
    df['Study_Hours'],
    df['Final_Marks'],
    c=df['Previous_Marks'],
    cmap='viridis',
    alpha=0.7,
    edgecolors='none'
)
plt.colorbar(scatter, label='Previous Marks')
plt.xlabel('Daily Study Hours')
plt.ylabel('Final Marks')
plt.title('Impact of Study Hours & Previous Foundation on Final Performance', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Train-Test Split and Model Benchmarking

In [ ]:
features = ['Study_Hours', 'Attendance', 'Previous_Marks']
target = 'Final_Marks'

X = df[features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Training Samples: {len(X_train)} | Testing Samples: {len(X_test)}')

In [ ]:
# Define and Train 4 Candidate Regression Models
models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(alpha=1.0),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42, max_depth=6),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42, max_depth=3)
}

results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = np.clip(model.predict(X_test), 0, 100)
    
    r2 = r2_score(y_test, preds)
    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    
    cv = cross_val_score(model, X, y, cv=5, scoring='r2').mean()
    
    results.append({
        'Model': name,
        'R2 Score': round(r2, 4),
        'RMSE': round(rmse, 2),
        'MAE': round(mae, 2),
        '5-Fold CV R2': round(cv, 4)
    })

benchmark_df = pd.DataFrame(results).sort_values('R2 Score', ascending=False)
benchmark_df

## 5. Model Evaluation & Residual Diagnostics

In [ ]:
# Visual Comparison of Models
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(data=benchmark_df, x='Model', y='R2 Score', palette='Blues_r', ax=ax[0])
ax[0].set_title('Model R² Comparison (Higher is Better)', fontweight='bold')
ax[0].set_ylim(0, 1.05)

sns.barplot(data=benchmark_df, x='Model', y='RMSE', palette='Reds_r', ax=ax[1])
ax[1].set_title('Model RMSE Error Comparison (Lower is Better)', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Actual vs. Predicted Marks Plot (Linear Regression)
lr_model = models['Linear Regression']
y_preds = lr_model.predict(X_test)

plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_preds, alpha=0.6, color='#2563eb', edgecolors='w', s=60)
plt.plot([y.min(), y.max()], [y.min(), y.max()], 'r--', lw=2, label='Ideal 45° Line')
plt.xlabel('Actual Marks')
plt.ylabel('Predicted Marks')
plt.title('Actual vs Predicted Marks — Linear Regression', fontsize=13, fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()

## 6. Interpretation: Regression Coefficients & Equation

In [ ]:
intercept = lr_model.intercept_
coefficients = pd.Series(lr_model.coef_, index=features)

print('Linear Regression Equation:')
print(f'Final Marks = {intercept:.2f} + (' + ' + '.join([f'{coef:.3f} * {feat}' for feat, coef in coefficients.items()]) + ')')

coefficients.plot(kind='barh', color='#0ea5e9', figsize=(8, 3))
plt.title('Feature Coefficients (Weight in Predicting Marks)', fontweight='bold')
plt.xlabel('Coefficient Value')
plt.show()

## 7. Interactive Scenario Test

In [ ]:
# Sample student test prediction
sample_student = pd.DataFrame([{
    'Study_Hours': 7.5,
    'Attendance': 90.0,
    'Previous_Marks': 78.0
}])

predicted_score = float(lr_model.predict(sample_student)[0])
print(f'Predicted Final Marks: {predicted_score:.2f} / 100')

if predicted_score >= 85:
    grade = 'A+ (Distinction)'
elif predicted_score >= 75:
    grade = 'A (Excellent)'
elif predicted_score >= 65:
    grade = 'B (Good)'
elif predicted_score >= 50:
    grade = 'C (Pass)'
else:
    grade = 'F (Needs Support)'

print(f'Assigned Grade Tier: {grade}')